# Packages

In [1]:
import cell2location

In [2]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl


from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

In [5]:
from pathlib import Path
import numpy as np 
from scipy import sparse # Dealing with adata data dyptes numbers


In [30]:
import pandas as pd

## SC: GPU

In [4]:
# SC for GPU
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (torch):", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))


Torch version: 2.5.1
CUDA available: True
CUDA version (torch): 12.4
GPU count: 1
Current device: 0
Device name: NVIDIA A100-PCIE-40GB


In [8]:
import torch, scvi
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.5.1 True NVIDIA A100-PCIE-40GB


# Data

## Locations

In [6]:
proj_folder   = Path("/home/janzules/spatial/CAR-T/data/cell2location")
input_folder  = proj_folder / "cell2location_inputs"

ref_loc       = input_folder / "ref_adata_c2l_model.h5ad"
spatial_loc   = input_folder / "spatial_sdata_c2l_model.h5ad"

# Output location
ref_run_name   = proj_folder / "reference_signatures"
run_name       = proj_folder / "cell2location_map"

# (results_folder / "gene_id_maps").mkdir(parents=True, exist_ok=True)
# (results_folder / "adata").mkdir(parents=True, exist_ok=True)
# (results_folder / "qc").mkdir(parents=True, exist_ok=True)

# spatial_prefix = results_folder / "gene_id_maps" / "spatial_symbols_to_ensembl"

# ref_prefix = results_folder / "gene_id_maps" / "ref_symbols_to_ensembl"

## Data Prep

In [24]:
sdata_c2l_model = sc.read_h5ad(spatial_loc)
adata_ref_c2l_model = sc.read_h5ad(ref_loc)

In [25]:

# spatial
sdata_c2l_model.X = sdata_c2l_model.X.tocsr()
sdata_c2l_model.X.data = sdata_c2l_model.X.data.astype(np.float32)

# reference
adata_ref_c2l_model.X = adata_ref_c2l_model.X.tocsr()
adata_ref_c2l_model.X.data = adata_ref_c2l_model.X.data.astype(np.float32)


In [31]:

# Creating the column that will match the sc reference
sdata_c2l_model.obs['treatment'] = sdata_c2l_model.obs['condition'].astype(str)

# Convert tumor_loc robustly: category -> string -> numeric
sdata_c2l_model.obs['tumor_loc'] = pd.to_numeric(
    sdata_c2l_model.obs['tumor_loc'].astype(str).str.strip(),
    errors='coerce'
).astype('Int64')  # nullable integer dtype

tumor_locations = [1, 2]

for tumor in tumor_locations:
    suffix = f"_Tu{tumor}"
    tum_loc_mask = sdata_c2l_model.obs["tumor_loc"].eq(tumor)

    sdata_c2l_model.obs.loc[tum_loc_mask, 'treatment'] = (
        sdata_c2l_model.obs.loc[tum_loc_mask, 'treatment']
        .str.replace(r"T72", "TAG72", regex=True)
        + suffix
    )


In [36]:
sdata_c2l_model.obs["batch_core"] = (
    sdata_c2l_model.obs["TMA"].astype(str) + "__" +
    sdata_c2l_model.obs["tissue"].astype(str)
).astype("category")

In [39]:
sdata_c2l_model.obs['batch_core'].unique()

['F07839__CyPSCA_1_1', 'F07840__CyPSCA_1_2', 'F08542__CyPSCA_2_4', 'F07840__CyT72_1_4', 'F08542__CyT72_2_3', ..., 'F07840__RTCyT72_2_4', 'F08543__CyT72_1_2', 'F08543__NoTx_2_4', 'F08543__RTCyPSCA_2_3', 'F08543__RTCyT72_1_1']
Length: 16
Categories (16, object): ['F07839__CyPSCA_1_1', 'F07839__NoTx_2_2', 'F07839__RTCyPSCA_1_4', 'F07839__RTCyT72_2_1', ..., 'F08543__CyT72_1_2', 'F08543__NoTx_2_4', 'F08543__RTCyPSCA_2_3', 'F08543__RTCyT72_1_1']

In [34]:
sdata_c2l_model.obs

,sample,cell_id,region,TMA,mouse,tissue,condition,tumor_loc,replicate_num,treatment
F07839_cellid_000000001-1,F07839,F07839_cellid_000000001-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1,CyPSCA_Tu1
F07839_cellid_000000002-1,F07839,F07839_cellid_000000002-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1,CyPSCA_Tu1
F07839_cellid_000000004-1,F07839,F07839_cellid_000000004-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1,CyPSCA_Tu1
F07839_cellid_000000005-1,F07839,F07839_cellid_000000005-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1,CyPSCA_Tu1
F07839_cellid_000000006-1,F07839,F07839_cellid_000000006-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1,CyPSCA_Tu1
...,...,...,...,...,...,...,...,...,...,...
F08543_cellid_000070488-1,F08543,F08543_cellid_000070488-1,RTCyT72_1_1_cell_boundaries,F08543,RTCyT72_1_1,RTCyT72_1_1,RTCyT72,1,1,RTCyTAG72_Tu1
F08543_cellid_000070489-1,F08543,F08543_cellid_000070489-1,RTCyT72_1_1_cell_boundaries,F08543,RTCyT72_1_1,RTCyT72_1_1,RTCyT72,1,1,RTCyTAG72_Tu1
F08543_cellid_000070490-1,F08543,F08543_cellid_000070490-1,RTCyT72_1_1_cell_boundaries,F08543,RTCyT72_1_1,RTCyT72_1_1,RTCyT72,1,1,RTCyTAG72_Tu1
F08543_cellid_000070491-1,F08543,F08543_cellid_000070491-1,RTCyT72_1_1_cell_boundaries,F08543,RTCyT72_1_1,RTCyT72_1_1,RTCyT72,1,1,RTCyTAG72_Tu1


# T: Exploring batch options

In [38]:
sdata_c2l_model.obs["batch_core"].value_counts().describe()

count       16.000000
mean     26706.000000
std      16387.594605
min       2747.000000
25%      16844.000000
50%      24401.500000
75%      33277.000000
max      65864.000000
Name: count, dtype: float64

In [37]:
cands = ["TMA", "sample", "tissue", "region", "mouse", "condition", "tumor_loc", "treatment", "batch_core"]
for k in cands:
    if k in sdata_c2l_model.obs.columns:
        vc = sdata_c2l_model.obs[k].value_counts()
        print(f"\n{k}: n={vc.size}, median={vc.median():.0f}, min={vc.min()}, max={vc.max()}")


TMA: n=4, median=94946, min=58530, max=178874

sample: n=4, median=94946, min=58530, max=178874

tissue: n=16, median=24402, min=2747, max=65864

region: n=16, median=24402, min=2747, max=65864

mouse: n=16, median=24402, min=2747, max=65864

condition: n=5, median=88824, min=49249, max=112113

tumor_loc: n=2, median=213648, min=162879, max=264417

treatment: n=10, median=38413, min=2747, max=97293

batch_core: n=16, median=24402, min=2747, max=65864
